In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark= SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
lending_df = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/user/itv024771/lc_raw_data.csv")

In [3]:
lending_df.createOrReplaceTempView("lendingclub_data")

In [4]:
from pyspark.sql.functions import *

In [5]:
new_lending_df = lending_df.withColumn("mem_id",sha2(concat_ws("||",*["emp_title","emp_length","home_ownership","annual_inc","verification_status","grade","sub_grade","zip_code","addr_state"]),256))

In [6]:
loan_data = ["mem_id","loan_amnt","funded_amnt","int_rate","installment","loan_status","term","total_pymnt","issue_d","purpose","title"]

In [7]:
loan_data_df = new_lending_df.select(*loan_data)

In [8]:
loan_data_df.show()

+--------------------+---------+-----------+--------+-----------+-----------+----------+-----------+--------+------------------+--------------------+
|              mem_id|loan_amnt|funded_amnt|int_rate|installment|loan_status|      term|total_pymnt| issue_d|           purpose|               title|
+--------------------+---------+-----------+--------+-----------+-----------+----------+-----------+--------+------------------+--------------------+
|415063bca49b803de...|     2500|       2500|   13.56|      84.92|    Current| 36 months|     167.02|Dec-2018|debt_consolidation|  Debt consolidation|
|c7fc9241e8e54e6d4...|    30000|      30000|   18.94|     777.23|    Current| 60 months|    1507.11|Dec-2018|debt_consolidation|  Debt consolidation|
|a244b73d2a8f0f8e7...|     5000|       5000|   17.97|     180.69|    Current| 36 months|     353.89|Dec-2018|debt_consolidation|  Debt consolidation|
|69524c0894efe3bd6...|     4000|       4000|   18.94|     146.51|    Current| 36 months|     286.71|

In [9]:
loan_data_df.createOrReplaceTempView("loan_tab")

In [10]:
spark.sql("select total_pymnt from loan_tab where total_pymnt is null").count()

1

In [11]:
loan_data_df.printSchema()

root
 |-- mem_id: string (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- term: string (nullable = true)
 |-- total_pymnt: double (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)



In [12]:
loan_data_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path","landing_club/loans_data") \
.save()

In [13]:
loan_schema = "mem_id string, loan_amnt float,funded_amnt integer, int_rate float,installment float, loan_status string, term string, total_pymnt float, issue_d string, purpose string, title string"

In [14]:
loan_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.schema(loan_schema) \
.load("/user/itv024771/landing_club/loans_data")

In [15]:
loan_raw_df.printSchema()

root
 |-- mem_id: string (nullable = true)
 |-- loan_amnt: float (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- int_rate: float (nullable = true)
 |-- installment: float (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- term: string (nullable = true)
 |-- total_pymnt: float (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)



In [16]:
loan_raw_ingest_df = loan_raw_df.withColumn("ingest_date",current_timestamp())

In [17]:
loan_raw_ingest_df.show()

+--------------------+---------+-----------+--------+-----------+-----------+---------+-----------+--------+------------------+--------------------+--------------------+
|              mem_id|loan_amnt|funded_amnt|int_rate|installment|loan_status|     term|total_pymnt| issue_d|           purpose|               title|         ingest_date|
+--------------------+---------+-----------+--------+-----------+-----------+---------+-----------+--------+------------------+--------------------+--------------------+
|415063bca49b803de...|   2500.0|       2500|   13.56|      84.92|    Current|36 months|     167.02|Dec-2018|debt_consolidation|  Debt consolidation|2026-03-27 08:51:...|
|c7fc9241e8e54e6d4...|  30000.0|      30000|   18.94|     777.23|    Current|60 months|    1507.11|Dec-2018|debt_consolidation|  Debt consolidation|2026-03-27 08:51:...|
|a244b73d2a8f0f8e7...|   5000.0|       5000|   17.97|     180.69|    Current|36 months|     353.89|Dec-2018|debt_consolidation|  Debt consolidation|20

In [18]:
loan_raw_ingest_df.createOrReplaceTempView("loans")

In [19]:
spark.sql("select count(*) from loans")

count(1)
220997


In [20]:
Checking where loan amt, intt rate, installments or loan status is null, drop the row:

SyntaxError: invalid syntax (<ipython-input-20-43f4bebfd654>, line 1)

In [21]:
spark.sql("select count(*) from loans where loan_amnt is null")

count(1)
0


In [22]:
cols_to_chk = ["loan_amnt","int_rate","installment","loan_status"]

In [23]:
loan_filtered_df = loan_raw_ingest_df.na.drop(subset=cols_to_chk)

In [24]:
loan_filtered_df.count()

220997

In [25]:
loan_filtered_df.createOrReplaceTempView("loans")

In [ ]:
I want term come in year format not month:

In [27]:
loan_tmod_df = loan_filtered_df.withColumn("term", (regexp_replace(col("term")," months", "").cast("int")/12).cast("int"))

In [28]:
loan_tmod_df.show()

+--------------------+---------+-----------+--------+-----------+-----------+----+-----------+--------+------------------+--------------------+--------------------+
|              mem_id|loan_amnt|funded_amnt|int_rate|installment|loan_status|term|total_pymnt| issue_d|           purpose|               title|         ingest_date|
+--------------------+---------+-----------+--------+-----------+-----------+----+-----------+--------+------------------+--------------------+--------------------+
|415063bca49b803de...|   2500.0|       2500|   13.56|      84.92|    Current|   3|     167.02|Dec-2018|debt_consolidation|  Debt consolidation|2026-03-27 08:53:...|
|c7fc9241e8e54e6d4...|  30000.0|      30000|   18.94|     777.23|    Current|   5|    1507.11|Dec-2018|debt_consolidation|  Debt consolidation|2026-03-27 08:53:...|
|a244b73d2a8f0f8e7...|   5000.0|       5000|   17.97|     180.69|    Current|   3|     353.89|Dec-2018|debt_consolidation|  Debt consolidation|2026-03-27 08:53:...|
|69524c089

In [29]:
loan_tmod_df.printSchema()

root
 |-- mem_id: string (nullable = true)
 |-- loan_amnt: float (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- int_rate: float (nullable = true)
 |-- installment: float (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- term: integer (nullable = true)
 |-- total_pymnt: float (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- ingest_date: timestamp (nullable = false)



In [30]:
loan_tmod_df.createOrReplaceTempView("loans")

In [31]:
spark.sql("select * from loans")

mem_id,loan_amnt,funded_amnt,int_rate,installment,loan_status,term,total_pymnt,issue_d,purpose,title,ingest_date
415063bca49b803de...,2500.0,2500,13.56,84.92,Current,3,167.02,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
c7fc9241e8e54e6d4...,30000.0,30000,18.94,777.23,Current,5,1507.11,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
a244b73d2a8f0f8e7...,5000.0,5000,17.97,180.69,Current,3,353.89,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
69524c0894efe3bd6...,4000.0,4000,18.94,146.51,Current,3,286.71,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
b4cf58748a1f31095...,30000.0,30000,16.14,731.78,Current,5,1423.21,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
9792c10a5738e0830...,5550.0,5550,15.02,192.45,Current,3,377.95,Dec-2018,credit_card,Credit card refin...,2026-03-27 08:55:...
25799584b2f48188e...,2000.0,2000,17.97,72.28,Current,3,141.56,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
e83de1407d94ea412...,6000.0,6000,13.56,203.79,Current,3,201.53,Dec-2018,credit_card,Credit card refin...,2026-03-27 08:55:...
d3250507ca2ea03cf...,5000.0,5000,17.97,180.69,Current,3,353.89,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...
0ba35020f6749f8a1...,6000.0,6000,14.47,206.44,Current,3,405.64,Dec-2018,debt_consolidation,Debt consolidation,2026-03-27 08:55:...


In [ ]:
Now want to check how many distinct values are there under purpose: 

In [32]:
spark.sql("select distinct(purpose) as dpprs from loans")

dpprs
other
small_business
debt_consolidation
credit_card
moving
debt_consol
vacation
renewable_energy
house
car


In [34]:
spark.sql("select purpose, count(*) as tpprs from loans group by purpose order by tpprs desc")

purpose,tpprs
debt_consolidation,121248
credit_card,60343
home_improvement,13302
other,12046
major_purchase,4004
medical,2465
car,1805
small_business,1725
vacation,1363
house,1348


In [38]:
loan_tmod_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv024771/landing_club/cleaned/loan_cleaned_data_pr") \
.save()